----
# SLH-DSA
----

In [5]:
import os

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.15.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs
import time
import numpy as np
import matplotlib.pyplot as plt
import csv
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.serialization import Encoding, NoEncryption, PrivateFormat, PublicFormat

benchmark_output_path = "slh_dsa_benchmark_results.txt"
csv_output_path = "slh_dsa_timing.csv"
rsa_benchmark_output_path = "rsa_4096_benchmark_results.txt"
rsa_csv_output_path = "rsa_4096_timing.csv"


def write_header(file_path, header_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(header_text + "\n")
        f.write("=" * len(header_text) + "\n")

write_header(benchmark_output_path, "SLH-DSA Benchmark Results")
write_header(rsa_benchmark_output_path, "RSA-4096 Benchmark Results")

csv_enabled = True
try:
    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms"])
    with open(rsa_csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms"])
except PermissionError:
    print(f"Warning: Cannot write to one or more CSV files (file may be open in another program). CSV logging disabled.")
    csv_enabled = False




def append_to_output(file_path, *lines):
    with open(file_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")


def append_to_csv(csv_path, metric, run, time_ms):
    if csv_enabled:
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms])



Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.15.0


In [6]:
# Test SLH-DSA availability
print("Testing SLH-DSA availability...")
print("\nAvailable SLH-DSA algorithms:")
slh_algs = [alg for alg in oqs.get_enabled_sig_mechanisms() if "SLH" in alg.upper()]
for alg in slh_algs:
    print(f"  ✓ {alg}")

if not slh_algs:
    print("No SLH-DSA algorithms enabled in this OQS build.")
else:
    print("\n✓ SLH-DSA setup successful!")

Testing SLH-DSA availability...

Available SLH-DSA algorithms:
  ✓ SLH_DSA_PURE_SHA2_128S
  ✓ SLH_DSA_PURE_SHA2_128F
  ✓ SLH_DSA_PURE_SHA2_192S
  ✓ SLH_DSA_PURE_SHA2_192F
  ✓ SLH_DSA_PURE_SHA2_256S
  ✓ SLH_DSA_PURE_SHA2_256F
  ✓ SLH_DSA_PURE_SHAKE_128S
  ✓ SLH_DSA_PURE_SHAKE_128F
  ✓ SLH_DSA_PURE_SHAKE_192S
  ✓ SLH_DSA_PURE_SHAKE_192F
  ✓ SLH_DSA_PURE_SHAKE_256S
  ✓ SLH_DSA_PURE_SHAKE_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2

In [7]:
# Benchmark SLH-DSA keypair generation for a representative parameter set
alg = "SLH_DSA_PURE_SHA2_256F"
if alg not in slh_algs:
    raise ValueError(f"Algorithm {alg} is not enabled. Choose one of: {slh_algs}")

print(f"Benchmarking {alg} keypair generation...")
runs = 20
times_ms = []
with oqs.Signature(alg) as sig:
    for run_index in range(runs):
        start = time.perf_counter()
        _ = sig.generate_keypair()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        times_ms.append(elapsed_ms)
        append_to_csv(csv_output_path, "keypair_generation", run_index + 1, elapsed_ms)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(times_ms):.3f} ms")
print(f"Min keypair time: {np.min(times_ms):.3f} ms")
print(f"Max keypair time: {np.max(times_ms):.3f} ms")
print("Times (ms):", [round(t, 3) for t in times_ms])

append_to_output(
    benchmark_output_path,
    "\nBenchmarking {alg} keypair generation...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(times_ms):.3f} ms",
    f"Min keypair time: {np.min(times_ms):.3f} ms",
    f"Max keypair time: {np.max(times_ms):.3f} ms",

    f"Times (ms): {[round(t, 3) for t in times_ms]}")

Benchmarking SLH_DSA_PURE_SHA2_256F keypair generation...
Algorithm: SLH_DSA_PURE_SHA2_256F
Runs: 20
Mean keypair time: 15.275 ms
Min keypair time: 9.847 ms
Max keypair time: 30.590 ms
Times (ms): [10.979, 10.34, 11.513, 9.882, 10.947, 12.108, 14.557, 10.239, 9.847, 22.018, 18.605, 17.641, 24.254, 17.678, 11.259, 22.524, 10.45, 9.99, 30.59, 20.085]


In [8]:
# Benchmark SLH-DSA signing time
message = b"Hello, world! This is a test message for SLH-DSA signing benchmark."

print(f"Benchmarking {alg} signing...")
runs = 20
sign_times_ms = []
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for run_index in range(runs):
        start = time.perf_counter()
        signature = sig.sign_with_ctx_str(message, context)
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        sign_times_ms.append(elapsed_ms)
        append_to_csv(csv_output_path, "signing", run_index + 1, elapsed_ms)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean signing time: {np.mean(sign_times_ms):.3f} ms")
print(f"Min signing time: {np.min(sign_times_ms):.3f} ms")
print(f"Max signing time: {np.max(sign_times_ms):.3f} ms")
print("Signing times (ms):", [round(t, 3) for t in sign_times_ms])

append_to_output(
    benchmark_output_path,
    "\nBenchmarking {alg} signing...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean signing time: {np.mean(sign_times_ms):.3f} ms",
    f"Min signing time: {np.min(sign_times_ms):.3f} ms",
    f"Max signing time: {np.max(sign_times_ms):.3f} ms",

    f"Signing times (ms): {[round(t, 3) for t in sign_times_ms]}")

Benchmarking SLH_DSA_PURE_SHA2_256F signing...
Algorithm: SLH_DSA_PURE_SHA2_256F
Runs: 20
Mean signing time: 275.551 ms
Min signing time: 231.460 ms
Max signing time: 378.886 ms
Signing times (ms): [357.578, 275.822, 293.457, 258.555, 378.886, 246.148, 257.267, 306.567, 270.462, 270.401, 258.092, 272.616, 253.445, 263.324, 253.382, 231.46, 232.615, 266.776, 271.619, 292.539]


In [9]:
# SLH-DSA Verification Benchmark
# Note: Verification is currently not working due to a possible bug in the oqs library for SLH-DSA algorithms.
# The regular verify and verify_with_ctx_str methods return False even with correct signatures.
# This may be due to the library version or implementation issues.
# Skipping verification benchmark for now.

In [10]:
# Measure SLH-DSA size metrics
print(f"Measuring {alg} size metrics...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    sk = sig.export_secret_key()
    context = b""
    signature = sig.sign_with_ctx_str(message, context)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(pk)} bytes")
print(f"Secret key size: {len(sk)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    benchmark_output_path,
    "\nMeasuring {alg} size metrics...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Public key size: {len(pk)} bytes",
    f"Secret key size: {len(sk)} bytes",

    f"Signature size: {len(signature)} bytes")

Measuring SLH_DSA_PURE_SHA2_256F size metrics...
Algorithm: SLH_DSA_PURE_SHA2_256F
Public key size: 64 bytes
Secret key size: 128 bytes
Signature size: 49856 bytes


In [11]:
# RSA-4096 comparison benchmark
rsa_message = b"Hello, world! This is a test message for RSA-4096 benchmark."
rsa_runs = 20

print("Benchmarking RSA-4096 keypair generation...")
rsa_keygen_times = []
for run_index in range(rsa_runs):
    start = time.perf_counter()
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=4096)
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    rsa_keygen_times.append(elapsed_ms)
    append_to_csv(rsa_csv_output_path, "keypair_generation", run_index + 1, elapsed_ms)

print(f"Runs: {rsa_runs}")
print(f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms")
print(f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms")
print(f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms")
print("Times (ms):", [round(t, 3) for t in rsa_keygen_times])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking RSA-4096 keypair generation...",
    f"Runs: {rsa_runs}",
    f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms",
    f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms",
    f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms",
    f"Times (ms): {[round(t, 3) for t in rsa_keygen_times]}")

print("Benchmarking RSA-4096 signing...")
rsa_sign_times = []
public_key = private_key.public_key()
for run_index in range(rsa_runs):
    start = time.perf_counter()
    signature = private_key.sign(
        rsa_message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    rsa_sign_times.append(elapsed_ms)
    append_to_csv(rsa_csv_output_path, "signing", run_index + 1, elapsed_ms)

print(f"Runs: {rsa_runs}")
print(f"Mean signing time: {np.mean(rsa_sign_times):.3f} ms")
print(f"Min signing time: {np.min(rsa_sign_times):.3f} ms")
print(f"Max signing time: {np.max(rsa_sign_times):.3f} ms")
print("Signing times (ms):", [round(t, 3) for t in rsa_sign_times])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking RSA-4096 signing...",
    f"Runs: {rsa_runs}",
    f"Mean signing time: {np.mean(rsa_sign_times):.3f} ms",
    f"Min signing time: {np.min(rsa_sign_times):.3f} ms",
    f"Max signing time: {np.max(rsa_sign_times):.3f} ms",
    f"Signing times (ms): {[round(t, 3) for t in rsa_sign_times]}")

print("Measuring RSA-4096 size metrics...")
rsa_private_bytes = private_key.private_bytes(
    encoding=Encoding.PEM,
    format=PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=NoEncryption(),
)
rsa_public_bytes = public_key.public_bytes(
    encoding=Encoding.PEM,
    format=PublicFormat.SubjectPublicKeyInfo,
)

print(f"Public key size: {len(rsa_public_bytes)} bytes")
print(f"Private key size: {len(rsa_private_bytes)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    rsa_benchmark_output_path,
    "\nMeasuring RSA-4096 size metrics...",
    f"Public key size: {len(rsa_public_bytes)} bytes",
    f"Private key size: {len(rsa_private_bytes)} bytes",
    f"Signature size: {len(signature)} bytes")

Benchmarking RSA-4096 keypair generation...
Runs: 20
Mean keypair time: 512.774 ms
Min keypair time: 133.919 ms
Max keypair time: 1097.818 ms
Times (ms): [316.192, 1002.843, 133.919, 481.548, 184.32, 239.472, 1097.818, 439.387, 747.262, 1078.978, 420.522, 206.382, 783.506, 248.459, 329.211, 422.227, 852.506, 391.767, 201.727, 677.436]
Benchmarking RSA-4096 signing...
Runs: 20
Mean signing time: 4.500 ms
Min signing time: 2.687 ms
Max signing time: 7.722 ms
Signing times (ms): [7.722, 4.671, 3.992, 3.597, 3.685, 6.242, 3.517, 2.687, 6.339, 3.071, 4.346, 3.772, 4.649, 6.279, 6.078, 4.045, 3.018, 4.783, 3.492, 4.021]
Measuring RSA-4096 size metrics...
Public key size: 800 bytes
Private key size: 3243 bytes
Signature size: 512 bytes
